[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ai-agents-certified/notebooks/day-01-langgraph-fundamentals.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · LangGraph Fundamentals — Graphs, Nodes, Edges, and State
**certified-journeys / ai-agents-certified** · Day 1 · Core Concepts

> **Goal for today:** Build and run a minimal two-node LangGraph graph that transforms text through state, and be able to explain what a graph, node, edge, and state object are and how they relate.

In [ ]:
%pip install -q langgraph langchain-openai

## Step 1 · What is LangGraph? Graphs, Nodes, and Edges

LangGraph models an agent as a **directed graph**. Each component has a clear role:

| Concept | What it is | Analogy |
|---------|------------|--------|
| **StateGraph** | The container — defines schema and wires everything together | A pipeline blueprint |
| **State** | A typed dict that travels through the graph — nodes read and write it | A shared scratchpad |
| **Node** | A Python function that receives state and returns an updated dict | A pipeline step |
| **Edge** | A directed connection from one node to another (or to END) | An arrow between steps |
| **START** | Virtual entry point — the first edge begins here | Front door |
| **END** | Virtual exit point — the graph halts when it reaches END | Back door |

The minimal lifecycle:
1. Define a **state schema** (what data flows)
2. Create a **StateGraph** with that schema
3. Add **nodes** (functions)
4. Add **edges** (connections)
5. **Compile** the graph → a runnable object
6. **Invoke** with an initial state → get back a final state dict

In [ ]:
# Imports — everything needed for a minimal graph
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# --- 1. Define the state schema ---
# TypedDict gives LangGraph the schema; each field is a slot in the shared scratchpad
class TextState(TypedDict):
    text: str          # the string being transformed
    step_log: list     # tracks which nodes have run

# --- 2. Define two node functions ---
# Each function receives the FULL current state dict and returns a PARTIAL update dict
def append_node(state: TextState) -> dict:
    """Node 1: appends '!!' to the text."""
    updated_text = state["text"] + "!!"
    updated_log  = state["step_log"] + ["append_node"]
    return {"text": updated_text, "step_log": updated_log}

def reverse_node(state: TextState) -> dict:
    """Node 2: reverses the text string."""
    updated_text = state["text"][::-1]   # Python slice trick for reversal
    updated_log  = state["step_log"] + ["reverse_node"]
    return {"text": updated_text, "step_log": updated_log}

# --- 3. Build the graph ---
graph_builder = StateGraph(TextState)

graph_builder.add_node("append", append_node)   # register node with a name
graph_builder.add_node("reverse", reverse_node)

# --- 4. Wire edges: START → append → reverse → END ---
graph_builder.add_edge(START, "append")
graph_builder.add_edge("append", "reverse")
graph_builder.add_edge("reverse", END)

# --- 5. Compile ---
graph = graph_builder.compile()

print("Graph compiled successfully")

### What just happened?

- **`StateGraph(TextState)`** creates a graph whose nodes all share the `TextState` schema.
- **`add_node`** registers a Python function under a string name — the name is what edges reference.
- **`add_edge`** defines execution order. `START` and `END` are built-in sentinels; you never define nodes for them.
- **`compile()`** validates the graph (no orphan nodes, no missing edges to END) and returns a `CompiledGraph` — the object you actually run.
- **Node return values are partial updates** — you only return the keys you changed; LangGraph merges them with the existing state.

## Step 2 · Running the Graph with `invoke`

A compiled graph exposes three execution methods:

| Method | Returns | Use when |
|--------|---------|----------|
| `invoke(input)` | Final state dict (all fields) | You want the end result |
| `stream(input)` | Generator of state snapshots | You want to watch each step |
| `ainvoke(input)` | Async final state | You're inside an async context |

Today we use `invoke`. The input must be a dict matching (a subset of) your state schema.

In [ ]:
# Run the graph
initial_state = {
    "text": "hello world",
    "step_log": []            # start with an empty log
}

result = graph.invoke(initial_state)

# Inspect the final state dict
print("Final text  :", result["text"])
print("Execution log:", result["step_log"])
print()
print("Full state dict:")
print(result)

### What just happened?

- `invoke` runs the graph **synchronously** — it blocks until the graph reaches `END`.
- The return value is the **complete final state**, not just what the last node returned. All fields are present.
- **`"hello world"` → `"hello world!!"` (append) → `"!!dlrow olleh"` (reverse)** — you can trace the transformation through `step_log`.
- Notice that neither node received a "result" argument — **state is the only communication channel** between nodes. This is the fundamental LangGraph pattern.

## Step 3 · Visualising the Graph Structure

LangGraph can emit a **Mermaid diagram** of your graph. In Colab you can render it inline; in terminal environments it prints the raw Mermaid source you can paste into [mermaid.live](https://mermaid.live).

Understanding the graph structure visually is important before the graphs get complex (conditional edges, cycles, subgraphs).

In [ ]:
# Print the Mermaid source — paste at https://mermaid.live to render
print(graph.get_graph().draw_mermaid())

In [ ]:
# In Colab: render inline with IPython display
try:
    from IPython.display import Image, display
    png_bytes = graph.get_graph().draw_mermaid_png()
    display(Image(png_bytes))
except Exception as e:
    # draw_mermaid_png requires the 'mermaid-py' package or a browser context
    print("PNG rendering not available here — see Mermaid source above.")
    print(f"Error detail: {e}")

### What just happened?

- `get_graph()` returns a `DrawableGraph` object that understands the compiled structure.
- `draw_mermaid()` emits a textual description: nodes as boxes, edges as arrows, `__start__` → `append` → `reverse` → `__end__`.
- **Tip:** Always draw your graph before running it on real data — a missing edge to END is a silent infinite loop, not a Python error.

## Step 4 · Using `stream` to Observe Each Step

`stream` yields a dict after **each node** executes. The key is the node name; the value is the partial state update that node returned. This is the best debugging tool for understanding how state evolves.

In [ ]:
print("=== Streaming execution trace ===")
print()

for step in graph.stream({"text": "langgraph", "step_log": []}):
    # step is {"node_name": {partial_update_dict}}
    node_name, update = next(iter(step.items()))
    print(f"Node '{node_name}' returned:")
    print(f"  text      = {update['text']!r}")
    print(f"  step_log  = {update['step_log']}")
    print()

### What just happened?

- **`stream` yields one dict per node execution** — each dict contains only what that node returned (the partial update), not the full accumulated state.
- You can use this to build progress UIs, debug ordering bugs, or feed intermediate results to other systems before the graph finishes.
- For the full accumulated state at each step, use `stream(mode="values")` — that gives you the complete state after each node.

## Step 5 · State Isolation and Why Nodes Return Dicts

A common first mistake is **mutating the input state** inside a node. LangGraph passes a copy of the current state, but mutating fields that are mutable (like lists) can cause subtle bugs. The correct pattern is always to return a new value.

This cell demonstrates the correct immutable pattern versus the problematic mutation pattern.

In [ ]:
# ✅ CORRECT: return a new list — never mutate state['step_log'] in-place
def safe_node(state: TextState) -> dict:
    new_log = state["step_log"] + ["safe_node"]  # creates a new list
    return {"step_log": new_log}

# ❌ RISKY: appending to the same list object
def risky_node(state: TextState) -> dict:
    state["step_log"].append("risky_node")   # mutates the list that LangGraph holds
    return {"step_log": state["step_log"]}

# Demonstrate the difference with a plain Python dict (no graph needed)
shared_state = {"text": "test", "step_log": ["existing_entry"]}

safe_result  = safe_node(shared_state)
print("After safe_node  — original step_log:", shared_state["step_log"])
print("After safe_node  — returned step_log:", safe_result["step_log"])

print()

risky_result = risky_node(shared_state)
print("After risky_node — original step_log:", shared_state["step_log"])  # mutated!
print("After risky_node — returned step_log:", risky_result["step_log"])

### What just happened?

- **`safe_node` uses `+` to create a new list** — the original `step_log` is unchanged; only the returned dict carries the new value.
- **`risky_node` appends in-place** — it mutates the same list object, which LangGraph's internal state snapshot also holds. With reducers (Day 2) this causes double-counting bugs.
- **Rule:** Treat your state values as immutable. Return new values; never mutate what you received. This rule is especially important for list and dict fields.

In [ ]:
# Challenge: Build a three-node pipeline that transforms a sentence
# Node 1 (uppercase_node): converts state['text'] to UPPER CASE
# Node 2 (word_count_node): adds a field state['word_count'] = number of words in text
# Node 3 (summary_node): sets state['summary'] = f"{word_count} words: {text[:30]}..."
#
# Extend the state schema below, implement the three nodes, wire the graph,
# and invoke it with "the quick brown fox jumps over the lazy dog".
# Expected final state keys: text, word_count, summary, step_log

class ChallengeState(TypedDict):
    text: str
    word_count: int
    summary: str
    step_log: list

# TODO: implement uppercase_node
# TODO: implement word_count_node
# TODO: implement summary_node
# TODO: build and compile the graph
# TODO: invoke and print the result

---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `StateGraph(Schema)` | Creates the graph; schema is a TypedDict subclass |
| Node function signature | `(state: Schema) -> dict` — receives full state, returns partial update |
| `add_edge(A, B)` | Every node that isn't END needs an outgoing edge |
| `compile()` | Validates and returns a `CompiledGraph` — the runnable object |
| `invoke(initial_state)` | Runs to END, returns the full final state dict |
| `stream(initial_state)` | Yields one `{node_name: partial_update}` dict per node execution |
| Immutability | Always return new values; never mutate the state you received |

> **Tip:** Start with the absolute minimum: one StateGraph, two nodes, one edge. Add complexity only after you can explain what state is and how it flows.

---
## What's next
**Day 2** → State Schemas and Reducers — TypedDict, `add_messages`, and custom reducers. You'll learn why list fields need reducers and how `MessagesState` tracks LLM conversation history.

Mark Day 1 complete in your [tracker](../index.html).